# Month 1 — Combined Dataset Baseline Pipeline: UCI 4-Site Cohort

This notebook runs the baseline framework on the **combined 4-site UCI Heart Disease dataset** (Cleveland, Hungarian, Switzerland, VA Long Beach; N=920 total).

### Steps:
1. **Load and Clean Combined Datasets**: Load all 4 sites, track `source_site` metadata, handle missing values (`?`), and binarize target variable (`0`: no disease, `1`: disease present).
2. **Missingness Report**: Generate missingness report across sites (noting high missingness in `ca` and `thal`).
3. **Preprocessing**: Impute and scale numerical features, impute and one-hot encode categorical features.
4. **Model Training & Hyperparameter Tuning**: Bayesian optimization (Optuna) for Random Forest, XGBoost, and AdaBoost.
5. **Soft-Voting Ensemble**: Combine the three tuned classifiers into a soft-voting ensemble.
6. **Nested Cross-Validation**: Evaluate all models and the ensemble using leak-free nested CV (5 outer folds, 3 inner folds).
7. **Explainability**: Generate SHAP feature importance summary and individual sample explanations saved with `combined_` prefix.


In [1]:
import os
import sys
import yaml
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure src module is in path
sys.path.append('..')

from src.preprocessing import load_combined_dataset, preprocess_data
from src.models import get_baseline_models
from src.optimization import optimize_all_models
from src.ensemble import build_voting_ensemble
from src.evaluation import evaluate_nested_cv, calculate_metrics
from src.explainability import generate_shap_explanations

# Load configuration
with open('../config.yaml', 'r') as f:
    config = yaml.safe_load(f)

print("Configuration loaded successfully.")


Configuration loaded successfully.


## 1. Data Loading, Concatenation & Missingness Analysis
Load Cleveland, Hungarian, Switzerland, and VA Long Beach datasets (N=920 total).


In [2]:
results_dir = '../' + config['results_dir']
df_combined = load_combined_dataset('../data', results_dir)

print(f"Combined Dataset Shape: {df_combined.shape}")
print("\nCohort Sample Sizes by Source Site:")
print(df_combined['source_site'].value_counts())

print("\nTarget Distribution (0 = No Disease, 1 = Heart Disease Present):")
print(df_combined['target'].value_counts())


Combined Dataset Shape: (920, 15)

Cohort Sample Sizes by Source Site:
source_site
cleveland        303
hungarian        294
va_long_beach    200
switzerland      123
Name: count, dtype: int64

Target Distribution (0 = No Disease, 1 = Heart Disease Present):
target
1    509
0    411
Name: count, dtype: int64


### Display Missingness Report Across Sites
Documenting missingness per column per site (CSV saved to `results/combined_missingness_report.csv`).


In [3]:
missing_report = pd.read_csv(f"{results_dir}/combined_missingness_report.csv", index_col=0)
print("=== Missingness Percentage per Column per Site ===")
display(missing_report)


=== Missingness Percentage per Column per Site ===


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
cleveland,0.0,0.0,0.0,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.32,0.66
hungarian,0.0,0.0,0.0,0.34,7.82,2.72,0.34,0.34,0.34,0.00,64.63,98.98,90.48
switzerland,0.0,0.0,0.0,1.63,100.00,60.98,0.81,0.81,0.81,4.88,13.82,95.93,42.28
va_long_beach,0.0,0.0,0.0,28.50,28.00,3.50,0.00,26.50,26.50,28.00,51.00,99.00,83.00
combined_overall,0.0,0.0,0.0,6.52,21.96,9.78,0.22,5.98,5.98,6.74,33.59,66.41,52.83


## 2. Preprocessing
Preprocess combined dataset using ColumnTransformer (median/mode imputation, scaling, one-hot encoding).


In [4]:
X_proc, y, preprocessor, feature_names = preprocess_data(df_combined)

print(f"Processed Feature Matrix Shape: {X_proc.shape}")
print(f"Extracted Feature Names ({len(feature_names)}):")
print(feature_names)


Processed Feature Matrix Shape: (920, 13)
Extracted Feature Names (13):
['age', 'sex', 'cp', 'trestbps', 'chol', 'fbs', 'restecg', 'thalach', 'exang', 'oldpeak', 'slope', 'ca', 'thal']


## 3. Bayesian Hyperparameter Optimization & Ensemble Setup
Tune Random Forest, XGBoost, and AdaBoost using Optuna on the full combined dataset, then construct a soft-voting ensemble.


In [5]:
tuned_models, best_params, best_scores = optimize_all_models(
    X_proc, y, cv=config['cv']['inner_folds'], n_trials=config['cv']['optuna_n_trials'], random_state=config['random_state']
)

print("=== Optuna Best Validation Scores on Combined Dataset (ROC-AUC) ===")
for m_name, score in best_scores.items():
    print(f"{m_name}: {score:.4f}")

# Fit tuned models on full combined dataset
for model in tuned_models.values():
    model.fit(X_proc, y)

# Build soft-voting ensemble
ensemble_model = build_voting_ensemble(tuned_models, voting='soft')
ensemble_model.fit(X_proc, y)
print("\nSoft-Voting Ensemble fitted on combined dataset successfully.")


=== Optuna Best Validation Scores on Combined Dataset (ROC-AUC) ===
random_forest: 0.8806
xgboost: 0.8849
adaboost: 0.8859



Soft-Voting Ensemble fitted on combined dataset successfully.


## 4. Leak-Free Nested Cross-Validation Evaluation
Run 5-fold outer / 3-fold inner nested CV on the 920-patient combined dataset.


In [6]:
print("Running Nested Cross-Validation evaluation on combined dataset...")
nested_results = evaluate_nested_cv(
    df_combined,
    target_col='target',
    outer_splits=config['cv']['outer_folds'],
    inner_splits=config['cv']['inner_folds'],
    n_trials=config['cv']['optuna_n_trials'],
    random_state=config['random_state']
)

model_names = ['random_forest', 'xgboost', 'adaboost', 'ensemble']
summary_rows = []

for model_name in model_names:
    metrics = nested_results[model_name]
    summary_rows.append({
        'Model': model_name.upper(),
        'Accuracy': f"{metrics['accuracy_mean']:.4f} ± {metrics['accuracy_std']:.4f}",
        'Precision': f"{metrics['precision_mean']:.4f} ± {metrics['precision_std']:.4f}",
        'Recall': f"{metrics['recall_mean']:.4f} ± {metrics['recall_std']:.4f}",
        'F1-Score': f"{metrics['f1_mean']:.4f} ± {metrics['f1_std']:.4f}",
        'ROC-AUC': f"{metrics['roc_auc_mean']:.4f} ± {metrics['roc_auc_std']:.4f}"
    })

summary_df = pd.DataFrame(summary_rows)
print("\n=== Combined Dataset Nested Cross-Validation Summary Results ===")
display(summary_df)


Running Nested Cross-Validation evaluation on combined dataset...



=== Combined Dataset Nested Cross-Validation Summary Results ===


,Model,Accuracy,Precision,Recall,F1-Score,ROC-AUC
0,RANDOM_FOREST,0.8043 ± 0.0223,0.8081 ± 0.0203,0.8488 ± 0.0461,0.8271 ± 0.0229,0.8815 ± 0.0150
1,XGBOOST,0.8196 ± 0.0111,0.8290 ± 0.0294,0.8528 ± 0.0486,0.8391 ± 0.0130,0.8858 ± 0.0167
2,ADABOOST,0.8087 ± 0.0210,0.8277 ± 0.0187,0.8272 ± 0.0435,0.8267 ± 0.0220,0.8837 ± 0.0180
3,ENSEMBLE,0.8130 ± 0.0074,0.8225 ± 0.0224,0.8468 ± 0.0416,0.8333 ± 0.0108,0.8861 ± 0.0154


## 5. Confusion Matrices
Display confusion matrices across outer cross-validation folds for each model on the combined dataset.


In [7]:
fig, axes = plt.subplots(2, 2, figsize=(10, 8))
axes = axes.flatten()

model_names = ['random_forest', 'xgboost', 'adaboost', 'ensemble']
for idx, model_name in enumerate(model_names):
    metrics = nested_results[model_name]
    cm = np.array(metrics['overall_confusion_matrix'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Greens', ax=axes[idx],
                xticklabels=['No Disease', 'Disease'],
                yticklabels=['No Disease', 'Disease'])
    axes[idx].set_title(f"Combined CM: {model_name.upper()}")
    axes[idx].set_xlabel('Predicted')
    axes[idx].set_ylabel('Actual')

plt.tight_layout()
plt.savefig(f"{results_dir}/combined_nested_cv_confusion_matrices.png", dpi=300)
plt.close('all')
print("Saved combined confusion matrices plot.")


Saved combined confusion matrices plot.


## 6. SHAP Explainability on Combined Dataset
Generate SHAP explanations for XGBoost predictions across the combined cohort, saving plots with `combined_` prefix.


In [8]:
xgb_model = tuned_models['xgboost']

explainer, shap_values = generate_shap_explanations(
    xgb_model,
    X_proc,
    feature_names,
    output_dir=results_dir,
    prefix='combined_xgboost',
    sample_indices=[0, 1]
)

print("SHAP explanations generated for combined dataset.")


SHAP explanations generated for combined dataset.
